**temp-upsert task**

**main table (retail_orders)**

In [0]:
%sql
CREATE OR REPLACE TABLE retail_orders (
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO retail_orders VALUES
(101,'Santhosh','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Suresh','Bangalore','Mobile',2,30000,'Placed'),
(103,'Jason','Mumbai','Headphones',3,2000,'Shipped');

num_affected_rows,num_inserted_rows
3,3


**inserting 2 new orders**

In [0]:
%sql
INSERT INTO retail_orders VALUES
(104,'Yadav','Chennai','Tablet',2,20000,'Placed'),
(105,'Abdullah','Delhi','Mobile',1,28000,'Placed');

num_affected_rows,num_inserted_rows
2,2


**updating laptop price to 78000**

In [0]:
%sql
UPDATE retail_orders
SET price = 78000
WHERE product = 'Laptop';

num_affected_rows
1


**deleting orders where quantity = 1**

In [0]:
%sql
DELETE FROM retail_orders
WHERE quantity = 1;

num_affected_rows
2


**temp table (incoming_orders)**

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT 101 AS order_id, 'santhosh' AS customer_name, 'Hyderabad' AS city, 'Laptop' AS product, 1 AS quantity, 78000 AS price, 'Placed' AS order_status
UNION ALL
SELECT 106, 'Bhawin', 'Kolkata', 'Mobile', 2, 26000, 'Placed'
UNION ALL
SELECT 107, 'Anand', 'Hyderabad', 'Headphones', 1, 2200, 'Placed';

In [0]:
%sql
MERGE INTO retail_orders AS target
USING incoming_orders AS source
ON target.customer_name = source.customer_name

WHEN MATCHED THEN
UPDATE SET
  target.city = source.city,
  target.product = source.product,
  target.quantity = source.quantity,
  target.price = source.price,
  target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT (
  order_id,
  customer_name,
  city,
  product,
  quantity,
  price,
  order_status
)
VALUES (
  source.order_id,
  source.customer_name,
  source.city,
  source.product,
  source.quantity,
  source.price,
  source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


In [0]:
%sql
select * from retail_orders;

order_id,customer_name,city,product,quantity,price,order_status
102,Suresh,Bangalore,Mobile,2,30000,Placed
103,Jason,Mumbai,Headphones,3,2000,Shipped
104,Yadav,Chennai,Tablet,2,20000,Placed
107,Anand,Hyderabad,Headphones,1,2200,Placed
101,santhosh,Hyderabad,Laptop,1,78000,Placed
106,Bhawin,Kolkata,Mobile,2,26000,Placed


In [0]:
%sql
select * from incoming_orders;

order_id,customer_name,city,product,quantity,price,order_status
101,santhosh,Hyderabad,Laptop,1,78000,Placed
106,Bhawin,Kolkata,Mobile,2,26000,Placed
107,Anand,Hyderabad,Headphones,1,2200,Placed
